In [1]:
import os
import shutil
import subprocess
from typing import Any

from dotenv import load_dotenv
from IPython.display import Markdown, display
from utils.agent_visualizer import (
    display_agent_response,
    print_activity,
    reset_activity_context,
    visualize_conversation,
)

from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient

# 02 - Observability Agent

이전 Notebook에서는 기본 research agent와 Chief of Staff multi-agent framework를 구축했습니다. 이미 강력한 agent이지만 수행할 수 있는 일에는 여전히 한계가 있었습니다. Web search agent는 internet 검색으로 제한되었고, Chief of Staff agent는 자체 filesystem과의 상호작용으로 제한되었습니다.

이는 큰 제약입니다. 실제 agent는 database, API, filesystem 및 기타 전문 service 같은 다른 system과 상호작용해야 하는 경우가 많습니다. [MCP(Model Context Protocol)](https://modelcontextprotocol.io/docs/getting-started/intro)는 AI-tool 통합용 open-source standard로, agent와 이런 외부 system을 쉽게 연결할 수 있게 합니다. 이 Notebook에서는 MCP server를 agent에 연결하는 방법을 살펴봅니다.

**MCP에 대해 더 자세히 알고 싶으신가요?** 종합적인 설정 지침, 구성 모범 사례, 문제 해결 방법은 [Claude Code MCP 문서](https://docs.claude.com/en/docs/claude-code/mcp)를 참고하세요.

## MCP Server 소개
### 1. Git MCP server

먼저 agent에 Git repository를 이해하고 다룰 수 있는 능력을 제공하겠습니다. [Git MCP server](https://github.com/modelcontextprotocol/servers/tree/main/src/git)를 추가하면 agent는 commit 기록을 살펴보고, 파일 변경을 확인하고, branch를 생성하며, commit까지 할 수 있는 Git 전용 tool 13개를 사용하게 됩니다. 이를 통해 agent는 수동적인 관찰자에서 개발 workflow의 능동적인 참여자로 전환됩니다. 이 예제에서는 Git tool만 사용해 repository 기록을 탐색하도록 agent를 구성합니다. 매우 단순하지만, 이를 알면 pull request 자동 생성, 코드 변화 pattern 분석 또는 여러 repository에 걸친 복잡한 Git workflow 관리 지원이 가능한 agent를 상상하기 어렵지 않습니다.

In [2]:
# Git repository root 가져오기(mcp_server_git은 유효한 git repo 경로 필요)
# os.getcwd()는 하위 디렉터리를 반환할 수 있으므로 실제 repo root를 찾음
git_executable = shutil.which("git")
if git_executable is None:
    raise RuntimeError("Git executable not found in PATH")

git_repo_root = subprocess.run(  # noqa: S603
    [git_executable, "rev-parse", "--show-toplevel"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

# Git MCP server 정의(pyproject.toml에서 uv sync로 설치됨)
git_mcp: dict[str, Any] = {
    "git": {
        "command": "uv",
        "args": ["run", "python", "-m", "mcp_server_git", "--repository", git_repo_root],
    }
}

In [3]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model="claude-opus-4-6",
        mcp_servers=git_mcp,
        allowed_tools=["mcp__git"],
        # disallowed_tools는 agent가 git command를 실행하는 Bash 대신 MCP tool만 사용하게 함
        disallowed_tools=["Bash", "Task", "WebSearch", "WebFetch"],
        permission_mode="acceptEdits",
    )
) as agent:
    await agent.query(
        "Explore this repo's git history and provide a brief summary of recent activity."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Using: mcp__git__git_log()
🤖 Using: mcp__git__git_status()
🤖 Using: mcp__git__git_branch()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__git__git_log()
🤖 Using: mcp__git__git_status()
🤖 Using: mcp__git__git_branch()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...


In [4]:
display(Markdown(f"\nResult:\n{messages[-1].result}"))


Result:
## Git Repository Summary

### Current Branch
You're on the **`upstream-contribution`** branch (up to date with origin), with `main` also available locally.

---

### Recent Commit Activity (Last ~5 Days)

| Date | Author | Summary |
|------|--------|---------|
| **Nov 27, 2025** | costiash | 3 commits enhancing the **Claude Agent SDK** - improved chief of staff agent, notebooks, observability agent, research agent, documentation, and utilities |
| **Nov 26, 2025** | Pedram Navid | Added GitHub issue templates, `/review-issue` command, `/add-registry` slash command, and new cookbook entries |
| **Nov 25, 2025** | Elie Schoppik | Renamed PTC notebook to `programmatic_tool_calling_ptc.ipynb` for clarity |
| **Nov 24, 2025** | henrykeetay | Added **tool search cookbook** |
| **Nov 24, 2025** | Alex Notov | Multiple merges consolidating cookbooks for Opus 4.5, dependency updates |
| **Nov 23, 2025** | Cal Rueb | Simplified crop tool notebook with Claude Agent SDK section |
| **Nov 23, 2025** | Pedram Navid | PR comment fixes and lint cleanup |

---

### Key Themes in Recent Development
1. **Claude Agent SDK enhancements** - Major work on agent implementations (research, chief of staff, observability agents)
2. **New cookbooks** - Tool search, crop tool, programmatic tool calling
3. **CI/CD improvements** - PR review workflows, issue templates, slash commands
4. **Documentation** - Added troubleshooting guides, codebase overviews

---

### Working Directory Status
There are **uncommitted changes** in your working directory:
- **22 modified files** (mostly in `claude_agent_sdk/`)
- **4 deleted files** (documentation files in `docs/`)
- **6 untracked files** (new reports, plans, VS Code config)

These changes appear to be further work on the Claude Agent SDK agents, notebooks, and utilities that haven't been staged or committed yet.

### 2. GitHub MCP server

이제 로컬 Git 작업에서 GitHub platform 전체 통합으로 발전시켜 보겠습니다. [공식 GitHub MCP server](https://github.com/github/github-mcp-server/tree/main)로 전환하면 agent는 issue 및 pull request 관리부터 CI/CD workflow 모니터링, 코드 보안 alert 분석까지 GitHub 생태계 전체와 상호작용하는 tool 100개 이상을 사용하게 됩니다. 이 server는 public 및 private repository 모두에서 작동하므로, 보통 여러 수동 단계가 필요한 복잡한 GitHub workflow를 agent가 자동화할 수 있습니다.

#### 1단계: GitHub Token 설정하기

GitHub Personal Access Token이 필요합니다. [여기](https://github.com/settings/personal-access-tokens/new)에서 발급받아 .env 파일에 ```GITHUB_TOKEN="<token>"``` 형식으로 넣으세요.
> 참고: Token을 발급받을 때 기본 옵션(즉, public repo, account permission 없음)으로 "Fine-grained" token을 선택하는 것이 이 demo를 실행하는 가장 쉬운 방법입니다.

이 예제에서는 컴퓨터에서 [Docker](https://www.docker.com/products/docker-desktop/)도 실행해야 합니다. GitHub MCP server가 보안과 격리를 위해 container 환경에서 실행되므로 Docker가 필요합니다.

**Docker 빠른 설정:**
- [docker.com](https://www.docker.com/products/docker-desktop/)에서 Docker Desktop을 설치합니다
- Docker가 실행 중인지 확인합니다(system tray에 Docker icon이 표시됩니다)
- Terminal에서 `docker --version`으로 확인합니다
- **문제 해결:** Docker가 시작되지 않으면 BIOS에서 virtualization이 활성화되어 있는지 확인하세요. 자세한 설정 지침은 [Docker 문서](https://docs.docker.com/get-docker/)를 참고하세요

#### 2단계: MCP server를 정의하고 agent loop 시작하기!

In [5]:
# GitHub MCP server 정의
load_dotenv(override=True)
github_mcp: dict[str, Any] = {
    "github": {
        "command": "docker",
        "args": [
            "run",
            "-i",
            "--rm",
            "-e",
            "GITHUB_PERSONAL_ACCESS_TOKEN",
            "ghcr.io/github/github-mcp-server",
        ],
        "env": {"GITHUB_PERSONAL_ACCESS_TOKEN": os.environ.get("GITHUB_TOKEN")},
    }
}

In [6]:
# Agent 실행
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model="claude-opus-4-6",
        mcp_servers=github_mcp,
        allowed_tools=["mcp__github"],
        # disallowed_tools는 agent가 gh CLI를 실행하는 Bash 대신 MCP tool만 사용하게 함
        disallowed_tools=["Bash", "Task", "WebSearch", "WebFetch"],
        permission_mode="acceptEdits",
    )
) as agent:
    await agent.query(
        "Search for the anthropics/claude-agent-sdk-python repository and give me a few key facts about it."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Using: mcp__github__search_repositories()
✓ Tool completed
🤖 Thinking...


In [7]:
display(Markdown(f"\nResult:\n{messages[-1].result}"))


Result:
Here are the key facts about the **anthropics/claude-agent-sdk-python** repository:

| Fact | Details |
|------|---------|
| **Full Name** | anthropics/claude-agent-sdk-python |
| **URL** | https://github.com/anthropics/claude-agent-sdk-python |
| **Language** | Python |
| **Stars** | ⭐ 3,357 |
| **Forks** | 🍴 435 |
| **Open Issues** | 149 |
| **Created** | June 11, 2025 |
| **Last Updated** | December 4, 2025 |
| **Default Branch** | main |
| **Visibility** | Public |
| **Archived** | No |

This is the official Python SDK for building Claude agents, maintained by Anthropic. It's quite popular with over 3,300 stars and has an active community with 435 forks. The repository is actively maintained (recently updated) and has a notable number of open issues (149), which suggests active development and community engagement.

## 실제 사용 사례: Observability agent

이제 이렇게 간단한 설정만으로도 스스로 복구하는 software system 역할의 agent를 만들 수 있습니다!

In [8]:
load_dotenv(override=True)

prompt = """Analyze the CI health for facebook/react repository.

Examine the most recent runs of the 'CI' workflow and provide:
1. Current status and what triggered the run (push, PR, schedule, etc.)
2. If failing: identify the specific failing jobs/tests and assess severity
3. If passing: note any concerning patterns (long duration, flaky history)
4. Recommended actions with priority (critical/high/medium/low)

Provide a concise operational summary suitable for an on-call engineer.
Do not create issues or PRs - this is a read-only analysis."""

github_mcp: dict[str, Any] = {
    "github": {
        "command": "docker",
        "args": [
            "run",
            "-i",
            "--rm",
            "-e",
            "GITHUB_PERSONAL_ACCESS_TOKEN",
            "ghcr.io/github/github-mcp-server",
        ],
        "env": {"GITHUB_PERSONAL_ACCESS_TOKEN": os.environ.get("GITHUB_TOKEN")},
    }
}

messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model="claude-opus-4-6",
        mcp_servers=github_mcp,
        allowed_tools=["mcp__github"],
        # 중요: 실제로 tool 사용을 제한하려면 disallowed_tools가 필요합니다.
        # 이것이 없으면 allowed_tools는 permission 요청만 제어하며 가용성은 제어하지 않습니다.
        # Agent는 여전히 Bash에 접근할 수 있습니다(MCP 대신 `gh` CLI 사용 가능).
        disallowed_tools=["Bash", "Task", "WebSearch", "WebFetch"],
        permission_mode="acceptEdits",
    )
) as agent:
    await agent.query(prompt)
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Using: mcp__github__get_file_contents()
🤖 Using: mcp__github__list_commits()
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__get_file_contents()
🤖 Using: mcp__github__get_file_contents()
🤖 Using: mcp__github__list_pull_requests()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__get_commit()
🤖 Using: mcp__github__get_commit()
🤖 Using: mcp__github__get_commit()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__sea

In [9]:
display(Markdown(f"\nResult:\n{messages[-1].result}"))


Result:
Based on my comprehensive analysis of the facebook/react repository CI infrastructure, here is the operational summary:

---

# CI Health Analysis: facebook/react

## Executive Summary
**Overall Status: 🟢 HEALTHY**

The React repository's CI appears to be in good health. Recent commits to `main` have been successfully merged, and active PRs show passing CodeSandbox builds.

---

## 1. CI Infrastructure Overview

### Primary Workflows
| Workflow | Trigger | Purpose |
|----------|---------|---------|
| `runtime_build_and_test.yml` | Push to main, PRs | Main CI - builds, tests, Flow checks |
| `shared_lint.yml` | Push to main, PRs | Prettier, ESLint, license checks |
| `compiler_typescript.yml` | PRs touching compiler | Compiler-specific tests |
| `devtools_regression_tests.yml` | PRs | DevTools testing |

### Test Matrix Scale
- **90 test shards** (18 configurations × 5 shards each)
- **50 build jobs** (25 workers × 2 release channels)
- **50 test-build shards** (5 configurations × 10 shards)
- Flow checks across multiple inline configs

---

## 2. Recent Main Branch Status

| Commit | Date | Description | Status |
|--------|------|-------------|--------|
| `bf1afad` | Dec 4, 2025 | [react-dom/server] Fix hanging on Deno | ✅ Merged |
| `0526c79` | Dec 3, 2025 | Update changelog with latest releases | ✅ Merged |
| `7dc903c` | Dec 3, 2025 | Patch FlightReplyServer (security fix) | ✅ Merged |
| `36df5e8` | Dec 2, 2025 | Allow building single release channel | ✅ Merged |

**Last 10 commits:** All successfully merged to main, indicating CI is passing.

---

## 3. Active PR CI Status

| PR | Title | CodeSandbox Status |
|----|-------|-------------------|
| #35267 | Fix spelling (behaviour → behavior) | 🟡 Pending (building) |
| #35238 | DevTools navigating commits hotkey | ✅ Success |
| #35287 | Compiler: Fix variable name issue | ✅ Success |
| #35278 | Add DevTools console suppress option | ✅ Success |
| #35226 | Fizz: Push stalled use() to ownerStack | ✅ Success |

---

## 4. Risk Assessment

### ✅ Positive Indicators
- **Main branch stable**: All recent commits merged successfully
- **No open CI failure issues**: Search returned zero CI-related open bugs
- **Active development**: Security patches and features landing regularly
- **PR builds passing**: Most open PRs show successful builds

### ⚠️ Areas to Monitor
- **Large test matrix**: 190+ parallel jobs mean potential for infrastructure flakiness
- **Playwright-based e2e tests**: Browser-based tests can be flaky (Flight fixtures, DevTools e2e)
- **Cache dependencies**: Multiple cache strategies (v6 keys) - cache misses could slow builds

### 📊 CI Complexity Metrics
- ~37KB workflow file for main CI (`runtime_build_and_test.yml`)
- Heavy parallelization with matrix strategies
- Multiple artifact upload/download operations

---

## 5. Recommended Actions

| Priority | Action | Rationale |
|----------|--------|-----------|
| **LOW** | Monitor PR #35267 | Currently building - verify completion |
| **LOW** | No immediate action required | Main branch healthy, PRs passing |
| **INFO** | Security patch merged Dec 3 | PR #35277 fixed critical security vuln in FlightReplyServer - verify downstream impact |

---

## 6. On-Call Notes

**TL;DR for On-Call Engineer:**
- 🟢 **CI is GREEN** - No action required
- Main branch is healthy with successful merges in last 24h
- All checked PRs showing green/passing status
- No open issues flagged for CI failures or flakiness
- Recent security patch (#35277) was successfully merged - monitor for any regressions

**If issues arise:**
1. Check GitHub Actions tab directly: `https://github.com/facebook/react/actions`
2. Key workflows to monitor: "(Runtime) Build and Test", "(Shared) Lint"
3. Caches use `v6` key prefix - if widespread failures, consider cache invalidation

---

*Analysis performed: December 4, 2025*
*Data sources: GitHub API (commits, PRs, status checks, workflow files)*

In [10]:
reset_activity_context()
visualize_conversation(messages)

In [11]:
reset_activity_context()
display_agent_response(messages)

### Module로 사용하는 Observability Agent

`observability_agent/agent.py` module은 observability pattern을 재사용 가능한 `send_query` function으로 감쌉니다. 내부적으로 `utils.agent_visualizer`의 공유 시각화 utility를 import하여 사용합니다:
- **`reset_activity_context()`**: 각 query 시작 시 자동으로 호출됩니다
- **`print_activity()`**: 실행 중 실시간 feedback을 제공합니다
- **`display_agent_response()`**: 최종 결과를 rendering합니다(`display_result` parameter로 제어)

따라서 최소한의 코드로 module을 사용할 수 있습니다:

In [12]:
# 변경 사항을 반영하도록 module reload(개발 중 유용)
from observability_agent.agent import send_query

# Module 내부에서 activity 표시, context 초기화, 결과 시각화를 처리함
result = await send_query(
    "Check the CI status for the last 2 runs in anthropics/claude-agent-sdk-python. Just do 3 tool calls, be efficient."
)

🤖 Using: mcp__github__list_commits()
✓ Tool completed
🤖 Using: mcp__github__get_commit()
🤖 Using: mcp__github__get_commit()
✓ Tool completed
✓ Tool completed
🤖 Thinking...


Commit,Message,Date,Status
2437035,chore: bump bundled CLI version to 2.0.58,"Dec 3, 2025 20:09 UTC",⚠️ No CI status available
9809fb6,chore: release v0.1.11 (#383),"Dec 3, 2025 19:42 UTC",⚠️ No CI status available


Multi-turn 대화는 원활하게 작동합니다. `continue_conversation=True`만 전달하세요:

In [13]:
# 예제 2: 더 깊은 모니터링을 위한 multi-turn 대화
result1 = await send_query("What's the current CI status for facebook/react?")

🤖 Using: mcp__github__list_pull_requests()
🤖 Using: mcp__github__list_commits()
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__get_commit()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...


PR,Title,Author,CI Status,Updated
#35287,[compiler] Fix JSX variable name issue,@kostya-gromov,🟢 Success,2h ago
#35285,[compiler][poc] Reuse ValidateExhaustiveDeps,@josephsavona,🔵 Draft,6h ago
#35284,[compiler] Fix hoisted primitives bug,@josephsavona,🟢 Success,7h ago
#35282,[compiler] Add effect deps validator,@jackpope,🟢 Success,15h ago
#35281,Improve legacy context warning,@Harshrj53,🟢 Success,20h ago


In [14]:
# 대화를 이어서 더 깊이 조사
result2 = await send_query(
    "Are there any flaky tests in the recent failures? You can only make one tool call.",
    continue_conversation=True,
)

🤖 Using: mcp__github__search_issues()
✓ Tool completed
🤖 Thinking...


Metric,Status
Open flaky test issues,0
Recent CI failures,None detected
Test stability,✅ Stable


## 결론

Claude Code SDK가 Model Context Protocol(MCP)을 통해 외부 system과 원활하게 통합되는 방법을 보여 주었습니다. Git MCP server를 통한 로컬 Git 작업에서 시작하여 GitHub 전용 tool 100개 이상을 사용하는 GitHub platform 전체 통합으로 점진적으로 확장했습니다. 이를 통해 agent는 로컬 assistant에서 workflow를 모니터링하고, CI/CD 장애를 분석하며, production system에 실행 가능한 insight를 제공하는 강력한 observability system으로 전환되었습니다.

MCP server를 agent에 연결하여 GitHub Actions workflow를 모니터링하고, 실제 장애와 보안 제약을 구별하며, test 실패를 상세히 분석하는 자율적인 observability system을 만들었습니다. 이 system은 agent가 수동적인 모니터링에서 지능적인 incident 대응으로 나아가 DevOps workflow에 능동적으로 참여할 수 있음을 보여 줍니다.

여기까지로 Claude Code SDK tutorial series의 여정을 일단 마칩니다. 단순한 research agent에서 정교한 multi-agent orchestration으로, 마지막으로 MCP를 통한 외부 system 통합으로 발전했습니다. 이 pattern들은 함께 governance, compliance, observability를 유지하면서 현실의 복잡성을 처리할 수 있는 production용 agentic system 구축의 기반을 제공합니다.

### 모든 Notebook에서 배운 내용

**Notebook 00(Research Agent)**
- `query()` 및 `ClaudeSDKClient`를 사용하는 SDK 핵심 기초
- WebSearch 및 Read를 통한 기본 tool 사용
- 단순한 agent loop 및 대화 관리

**Notebook 01(Chief of Staff)**
- 고급 기능: memory, output style, planning mode
- Subagent를 통한 multi-agent 조정
- Hook 및 custom command를 통한 governance
- Enterprise에 사용할 수 있는 agent architecture

**Notebook 02(Observability Agent)**
- MCP server를 통한 외부 system 통합
- 실시간 모니터링 및 incident 대응
- Production workflow 자동화
- 확장 가능한 agent 배포 pattern

세 agent 모두의 전체 구현은 각 디렉터리(`research_agent/`, `chief_of_staff_agent/`, `observability_agent/`)에 있으며, production system 통합을 위한 참고 사례로 사용할 수 있습니다.